In [ ]:
from DataProcessor import DataProcessor
import matplotlib.pyplot as plt
import numpy as np

DEFAULT_DURATION = 60
plot_count = 10 
dp = DataProcessor("rawdata/X22/Excersice_2")
files = ['rawdata/X22/Excersice_2/EX2_gehen1.pickle', 'rawdata/X22/Excersice_2/EX2_gehen2.pickle', 'rawdata/X22/Excersice_2/EX2_gehen3.pickle', 
         'rawdata/X22/Excersice_2/EX2_gehen4.pickle', 'rawdata/X22/Excersice_2/EX2_gehen5.pickle'];
titles = ['normales Gehen 1', 'normales Gehen 2', 'normales Gehen 3', 'normales Gehen 4', 'normales Gehen 5'];
fs = dp.fs


def find_activity_start(acc_x, acc_y, acc_z, fs, threshold, window_seconds):
    """
    Findet den Index ab dem die Bewegung beginnt.
    Berechnet die Gesamtbeschleunigung (ohne Gravitation) und sucht
    den ersten Moment wo die Energie einen Schwellwert überschreitet.
    """
    window = int(window_seconds * fs)
    
    # Mittlere Beschleunigung (Gravitations-Offset) entfernen
    acc_x_centered = acc_x - np.mean(acc_x)
    acc_y_centered = acc_y - np.mean(acc_y)
    acc_z_centered = acc_z - np.mean(acc_z)
    
    # Betrag der Gesamtbeschleunigung
    magnitude = np.sqrt(acc_x_centered**2 + acc_y_centered**2 + acc_z_centered**2)
    
    # Gleitender Mittelwert (lokale Energie)
    energy = np.convolve(magnitude, np.ones(window)/window, mode='same')
    
    # Ersten Index suchen wo Energie > Schwellwert
    indices = np.where(energy > threshold)[0]
    
    if len(indices) == 0:
        return 0  # kein Schwellwert gefunden → von Anfang an
    return indices[0]

#Hilfsfunktion: eine Messung laden und Zeit + alle drei Achsen zurückgeben
def calc_time_and_axis_values():
    
    treshold = 0.1
    window_seconds = 0.1
    
     #Zeitvektor t (in Sekunden) aus dem Acceleration-DataFrame holen
    t_acc = dp.dfAcc["t"].values
    t_gyr = dp.dfGyr["t"].values
        
    #x-, y- und z-Komponente der Beschleunigung holen
    x_acc = dp.dfAcc["x"].values
    y_acc = dp.dfAcc["y"].values
    z_acc = dp.dfAcc["z"].values
    
    x_gyr = dp.dfGyr["x"].values
    y_gyr = dp.dfGyr["y"].values
    z_gyr = dp.dfGyr["z"].values
    
    
    #Analyse erstes sample, dass Bewegung beinhaltet
    start_idx = find_activity_start(x_acc, y_acc, z_acc, fs, treshold, window_seconds)
    
    #Anzahl samples berechnen
    num_samples_1 = int(DEFAULT_DURATION*fs)
    #num_samples_2= int(DEFAULT_DURATION*fs)
    
    #letzes sample berechnen
    end_idx = min(start_idx + num_samples_1, len(x_acc))
    
    #Werte auf 
    t_acc = t_acc[start_idx:end_idx]
    
    #Setzen des ersten Bewegungsverts auf index 0 der Plot-Achse
    t_acc = t_acc - t_acc[0]
    
    x_acc = x_acc[start_idx:end_idx]
    y_acc = y_acc[start_idx:end_idx]
    z_acc = z_acc[start_idx:end_idx]
    
    t_gyr = t_gyr[start_idx:end_idx]
    t_gyr = t_gyr - t_gyr[0]
    
    x_gyr = x_gyr[start_idx:end_idx]
    y_gyr = y_gyr[start_idx:end_idx]
    z_gyr = z_gyr[start_idx:end_idx]
    
    return t_acc, x_acc, y_acc, z_acc, t_gyr, x_gyr, y_gyr, z_gyr

loop_count = int(plot_count/2)

for i in range(loop_count): 
    fileName = files[i]
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
            
    t1, acc_x, acc_y, acc_z, t2, gyr_x, gyr_y, gyr_z =  calc_time_and_axis_values()
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)    
    #Drei Unterplots untereinander erstellen   
    axes[0].plot(t1, acc_x, label="x")
    axes[0].plot(t1, acc_y, label="y")
    axes[0].plot(t1, acc_z, label="z")
    axes[0].set_ylabel("Acceleration [g]")
    axes[0].set_title(titles[i])
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(t2, gyr_x, label="x")
    axes[1].plot(t2, gyr_y, label="y")
    axes[1].plot(t2, gyr_z, label="z")
    axes[1].set_ylabel("Gyros. [deg/s]")
    axes[1].legend()
    axes[1].grid(True)
    
    axes[1].set_xlabel("Zeit [s]")
    plt.tight_layout()
plt.show()

c:\Users\elyes\anaconda3\envs\DSP_Projekt\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>